In [1]:
# Celda: separación automática OLD vs NEW tests + comparativa rápida

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
import os
from datetime import datetime

# Crear/Reusar carpeta de análisis con fecha (persistente en esta sesión)
def _get_or_create_analysis_dir():
    root = Path("../data/backtesting")
    env_key = "ANALYSIS_BACKTESTING_DIR"
    existing = os.environ.get(env_key)
    if existing:
        d = Path(existing)
        d.mkdir(parents=True, exist_ok=True)
        return d
    stamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
    d = root / f"analisis backtesting {stamp}"
    d.mkdir(parents=True, exist_ok=True)
    os.environ[env_key] = str(d)
    return d

OUT = _get_or_create_analysis_dir()
OUT.mkdir(parents=True, exist_ok=True)
PLOTS = OUT / "before_after_plots"
PLOTS.mkdir(parents=True, exist_ok=True)

# Usa df_csvs ya cargado
df = df_csvs.copy()

# Asegurar timestamp y ganancia numérica
if "timestamp" in df.columns:
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
else:
    # crear índice temporal artificial si no existe
    df["timestamp"] = pd.to_datetime(df.index, unit="s", errors="coerce")

df["ganancia_num"] = pd.to_numeric(df.get("ganancia"), errors="coerce").fillna(0.0)

# Detectar punto de cambio: percentil alto (p99) como proxy de nuevas pruebas con ganancias grandes
p99 = df["ganancia_num"].quantile(0.99) if not df["ganancia_num"].empty else 0.0
cands = df[df["ganancia_num"] >= p99]
if not cands.empty:
    cutoff = cands["timestamp"].min()
else:
    # si no hay outliers, toma el 80% temporal como cutoff (último 20% como 'nuevo')
    cutoff = df["timestamp"].quantile(0.80)

# Split
df_old = df[df["timestamp"] < cutoff].sort_values("timestamp").reset_index(drop=True)
df_new = df[df["timestamp"] >= cutoff].sort_values("timestamp").reset_index(drop=True)

def quick_summary(d):
    s = {}
    gan = d["ganancia_num"].dropna()
    s["count"] = int(len(gan))
    s["mean"] = float(gan.mean()) if len(gan)>0 else None
    s["median"] = float(gan.median()) if len(gan)>0 else None
    s["std"] = float(gan.std()) if len(gan)>0 else None
    s["p1"] = float(gan.quantile(0.01)) if len(gan)>0 else None
    s["p99"] = float(gan.quantile(0.99)) if len(gan)>0 else None
    s["wins"] = int((gan>0).sum()) if len(gan)>0 else 0
    s["winrate"] = float((gan>0).sum()/len(gan)) if len(gan)>0 else None
    s["profit_factor"] = None
    pos = gan[gan>0].sum() if len(gan)>0 else 0.0
    neg = gan[gan<=0].abs().sum() if len(gan)>0 else 0.0
    if neg!=0:
        s["profit_factor"] = float(pos/neg)
    elif pos>0:
        s["profit_factor"] = float(np.inf)
    return s

summary = {
    "cutoff": str(cutoff),
    "old": quick_summary(df_old),
    "new": quick_summary(df_new),
    "global": quick_summary(df)
}

# Guardar JSON y CSVs
with open(OUT / "before_after_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

df_old.to_csv(OUT / "operaciones_antes_de_cambio.csv", index=False)
df_new.to_csv(OUT / "operaciones_despues_de_cambio.csv", index=False)

# Graficos comparativos
plt.figure(figsize=(10,4))
sns.histplot(df_old["ganancia_num"], color="gray", label="antes", bins=80, stat="density", alpha=0.6)
sns.histplot(df_new["ganancia_num"], color="green", label="después", bins=80, stat="density", alpha=0.6)
plt.legend()
plt.title("Distribución de ganancia: Antes vs Después (auto-cut)")
plt.tight_layout()
plt.savefig(PLOTS / "hist_before_after.png")
plt.close()

# Equity curves acumuladas
plt.figure(figsize=(10,5))
if not df_old.empty:
    plt.plot(df_old["ganancia_num"].cumsum().values, label=f"antes ({len(df_old)} ops)")
if not df_new.empty:
    plt.plot(df_new["ganancia_num"].cumsum().values, label=f"después ({len(df_new)} ops)")
plt.legend()
plt.title("Equity acumulada: Antes vs Después")
plt.xlabel("Operación")
plt.ylabel("Equity (suma ganancia)")
plt.tight_layout()
plt.savefig(PLOTS / "equity_before_after.png")
plt.close()

# Actualizar reflexiones
ref = Path("../data/logs/reflexiones_bot.txt")
note = f"\n[Auto] Split detected at {cutoff}. Ops antes: {summary['old']['count']}, después: {summary['new']['count']}. Summary saved to {OUT}/before_after_summary.json\n"
ref.write_text(ref.read_text(encoding="utf-8") + note if ref.exists() else note, encoding="utf-8")

print("Resumen before/after generado. Archivos creados en:", OUT)
print("Revisa:", OUT / "before_after_summary.json")

NameError: name 'df_csvs' is not defined

In [ ]:
# CELDA ADICIONAL: Limpieza avanzada, métricas robustas, inspección de outliers, equity plots y export final

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
import os
import re
from datetime import datetime

# Config: carpeta de análisis con fecha (común a todas las celdas)
def _get_or_create_analysis_dir():
    root = Path("../data/backtesting")
    env_key = "ANALYSIS_BACKTESTING_DIR"
    existing = os.environ.get(env_key)
    if existing:
        d = Path(existing)
        d.mkdir(parents=True, exist_ok=True)
        return d
    stamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
    d = root / f"analisis backtesting {stamp}"
    d.mkdir(parents=True, exist_ok=True)
    os.environ[env_key] = str(d)
    return d

ANALYSIS_DIR = _get_or_create_analysis_dir()
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = (ANALYSIS_DIR / "plots"); PLOTS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR = Path("../data/logs"); LOGS_DIR.mkdir(parents=True, exist_ok=True)

# También necesitamos conocer el RUN_DIR de los datos fuente
RUN_DIR = Path("../data/backtesting/2025-09-07_00-53")
if not RUN_DIR.exists():
    raise RuntimeError(f"Run dir no existe: {RUN_DIR}. Verifica la ruta relativa desde notebooks/.")

# ----- Helpers de limpieza/conversión -----
def parse_numeric_series(s: pd.Series) -> pd.Series:
    """Convierte strings con %, comas decimales o separadores en float."""
    s2 = s.astype(str).str.strip()
    # quitar %
    s2 = s2.str.replace('%', '', regex=False)
    # si no hay punto pero hay coma, considerar coma como decimal
    mask_coma_decimal = s2.str.contains(',', regex=False) & ~s2.str.contains('.', regex=False)
    s2 = s2.where(~mask_coma_decimal, s2.str.replace(',', '.', regex=False))
    # remover separadores de miles comunes
    s2 = s2.str.replace('\u00A0', '', regex=False)  # NBSP
    return pd.to_numeric(s2, errors='coerce')

def coalesce_profit_cols(df: pd.DataFrame) -> pd.Series:
    """Elige la mejor columna de PnL disponible y la devuelve como float."""
    candidates = [
        'ganancia', 'pnl', 'PnL', 'profit', 'profit_usd', 'resultado', 'retorno', 'retorno_usd', 'retorno_pct'
    ]
    present = [c for c in candidates if c in df.columns]
    if not present:
        return pd.Series([np.nan]*len(df))
    series = None
    for c in present:
        ser = df[c]
        if ser.dtype == 'O':
            ser_num = parse_numeric_series(ser)
        else:
            ser_num = pd.to_numeric(ser, errors='coerce')
        series = ser_num if series is None else series.fillna(ser_num)
    return series

def parse_duration_to_minutes(s: pd.Series) -> pd.Series:
    if s.dtype != 'O':
        return pd.to_numeric(s, errors='coerce')
    if s.astype(str).str.contains(':').any():
        td = pd.to_timedelta(s, errors='coerce')
        return td.dt.total_seconds() / 60.0
    return pd.to_numeric(s, errors='coerce')

# ----- 0. Carga de datos desde el run -----
snapshot_file = RUN_DIR / "resultados_backtesting_snapshot.csv"
use_trades_override = False  # si quieres forzar trades como fuente primaria, pon True

if snapshot_file.exists() and not use_trades_override:
    df_all = pd.read_csv(snapshot_file)
else:
    dfs = []
    patterns = [
        "*/trades.csv",
        "*/*with_equity.csv",
        "*/resultados_*_30d.csv",
    ]
    for pattern in patterns:
        for p in RUN_DIR.glob(pattern):
            try:
                d = pd.read_csv(p)
                if "symbol" not in d.columns:
                    d["symbol"] = p.parent.name
                if "periodo" not in d.columns:
                    name = p.name
                    d["periodo"] = ("30d" if "30d" in name else "NA")
                dfs.append(d)
            except Exception as e:
                print(f"Error leyendo {p}: {e}")
    if not dfs:
        raise RuntimeError(f"No se encontraron CSVs dentro de {RUN_DIR} para analizar.")
    df_all = pd.concat(dfs, ignore_index=True)

# Asegurar que existen columnas clave mínimas
for col in ("symbol", "periodo"):
    if col not in df_all.columns:
        raise RuntimeError(f"Columna requerida faltante en datos: {col}")

# ----- 1. Normalización y validación de columnas -----
time_cols = [c for c in df_all.columns if c.lower() in ("timestamp","time","fecha","date","datetime","ts","open_time","close_time")]
time_col = time_cols[0] if time_cols else None

if time_col:
    df_all[time_col] = pd.to_datetime(df_all[time_col], errors="coerce")
    df_all = df_all.sort_values(time_col).reset_index(drop=True)
else:
    print("⚠️ No se encontró columna de tiempo conocida. Equity y drawdown se calcularán en orden de aparición.")

# Convertir columnas numéricas
if 'ganancia' in df_all.columns:
    raw_g = df_all['ganancia']
    gnum_try = parse_numeric_series(raw_g) if raw_g.dtype == 'O' else pd.to_numeric(raw_g, errors='coerce')
else:
    gnum_try = pd.Series([np.nan]*len(df_all))

pnl_coalesced = coalesce_profit_cols(df_all)
nan_ratio = float(gnum_try.isna().mean()) if len(gnum_try)>0 else 1.0
if nan_ratio > 0.5 and pnl_coalesced.notna().any():
    df_all['ganancia_num'] = pnl_coalesced
else:
    df_all['ganancia_num'] = gnum_try

if 'duracion' in df_all.columns:
    df_all['duracion_num'] = parse_duration_to_minutes(df_all['duracion'])
else:
    df_all['duracion_num'] = pd.to_numeric(df_all.get('duracion'), errors='coerce')

df_all['precio_num'] = pd.to_numeric(df_all.get('precio'), errors='coerce')

if 'tipo' not in df_all.columns:
    df_all['tipo'] = ""
else:
    df_all['tipo'] = df_all['tipo'].astype(str)

# Eliminar duplicados exactos
df_all = df_all.drop_duplicates().reset_index(drop=True)

# Reporte y fallback a trades si sigue habiendo demasiados NaN en ganancia
nan_gan = df_all['ganancia_num'].isna().sum()
if nan_gan:
    print(f"Advertencia: {nan_gan} filas con ganancia no convertible (NaN). Se excluirán de métricas numéricas puntuales.")

if (len(df_all) > 0) and (df_all['ganancia_num'].isna().mean() > 0.5):
    dfs_tr = []
    for p in RUN_DIR.glob('*/trades.csv'):
        try:
            d = pd.read_csv(p)
            d['symbol'] = p.parent.name
            if 'periodo' not in d.columns:
                d['periodo'] = '30d'
            d['ganancia_num'] = coalesce_profit_cols(d)
            if 'duracion' in d.columns:
                d['duracion_num'] = parse_duration_to_minutes(d['duracion'])
            dfs_tr.append(d)
        except Exception as e:
            print(f"Error leyendo trades {p}: {e}")
    if dfs_tr:
        df_all = pd.concat(dfs_tr, ignore_index=True)
        print("Fuente primaria cambiada a trades.csv por alto porcentaje de NaN en 'ganancia'.")
    else:
        print("No se pudo usar trades.csv como fallback; se continúa con datos disponibles.")

# Excluir filas sin PnL numérico para métricas
df_all = df_all.dropna(subset=['ganancia_num']).reset_index(drop=True)

# ----- 2. Funciones auxiliares métricas robustas -----
def trimmed_mean(arr, proportion=0.05):
    a = np.sort(np.asarray(arr))
    n = len(a)
    if n == 0:
        return None
    k = int(np.floor(n * proportion))
    if n - 2*k <= 0:
        return float(np.mean(a))
    return float(a[k:n-k].mean())

def iqr(arr):
    s = pd.Series(arr).dropna()
    if s.empty:
        return None
    return float(s.quantile(0.75) - s.quantile(0.25))

def sharpe_like(arr):
    s = pd.Series(arr).dropna()
    if s.empty or s.std() == 0:
        return None
    return float(s.mean() / s.std())

def profit_factor(pos, neg):
    total_pos = np.sum(pos) if len(pos)>0 else 0.0
    total_neg = np.abs(np.sum(neg)) if len(neg)>0 else 0.0
    if total_neg == 0:
        return (np.inf if total_pos>0 else None)
    return float(total_pos / total_neg)

# ----- 3. Por grupo: métricas, equity y drawdown -----
grouped_results = {}
for (sym, per), g in df_all.groupby(['symbol','periodo']):
    g_sorted = g.copy()
    if time_col and time_col in g_sorted.columns:
        g_sorted = g_sorted.sort_values(time_col)
    gan = g_sorted['ganancia_num'].astype(float)
    pos = gan[gan>0]
    neg = gan[gan<=0]
    ops = len(gan)
    wins = int((gan>0).sum())
    losses = int((gan<=0).sum())
    denom = ops - int((gan==0).sum())
    winrate = float(wins / denom) if denom > 0 else None
    pf = profit_factor(pos, neg)

    expect = float(gan.mean()) if ops>0 else None
    median = float(np.median(gan)) if ops>0 else None
    tmean = trimmed_mean(gan, 0.05) if ops>0 else None
    iqr_v = iqr(gan)
    sharpe_v = sharpe_like(gan)

    # equity y drawdown
    equity = gan.cumsum().ffill().fillna(0.0)
    running_max = equity.cummax()
    drawdown = (running_max - equity) / running_max.replace(0, np.nan)
    drawdown = drawdown.fillna(0.0)
    max_dd = float(drawdown.max()) if not drawdown.empty else 0.0

    grouped_results[f"{sym}_{per}"] = {
        'symbol': sym,
        'periodo': per,
        'operaciones': int(ops),
        'wins': wins,
        'losses': losses,
        'winrate': winrate,
        'profit_factor': pf,
        'expectancy': expect,
        'median': median,
        'trimmed_mean_5pct': tmean,
        'iqr': iqr_v,
        'sharpe_like': sharpe_v,
        'max_drawdown_pct': max_dd,
        'ganancia_summary': {
            'mean': float(gan.mean()) if ops>0 else None,
            'median': median,
            'std': float(gan.std()) if ops>0 else None,
            'min': float(gan.min()) if ops>0 else None,
            'max': float(gan.max()) if ops>0 else None,
            'p1': float(gan.quantile(0.01)) if ops>0 else None,
            'p99': float(gan.quantile(0.99)) if ops>0 else None,
        },
        'duracion_summary': {
            'mean': float(g_sorted['duracion_num'].mean()) if 'duracion_num' in g_sorted.columns else None,
            'median': float(g_sorted['duracion_num'].median()) if 'duracion_num' in g_sorted.columns else None,
        }
    }

# ----- 4. Resumen global -----
gan_all = df_all['ganancia_num']
pos_all = gan_all[gan_all>0]
neg_all = gan_all[gan_all<=0]
pf_global = profit_factor(pos_all, neg_all)
winrate_global = float((gan_all>0).sum() / (len(gan_all[gan_all!=0]))) if len(gan_all[gan_all!=0])>0 else None

global_summary = {
    'operations_total': int(len(gan_all)),
    'wins_total': int((gan_all>0).sum()),
    'losses_total': int((gan_all<=0).sum()),
    'winrate': winrate_global,
    'profit_factor': pf_global,
    'mean': float(gan_all.mean()) if len(gan_all)>0 else None,
    'median': float(gan_all.median()) if len(gan_all)>0 else None,
    'std': float(gan_all.std()) if len(gan_all)>0 else None,
    'p1': float(gan_all.quantile(0.01)) if len(gan_all)>0 else None,
    'p99': float(gan_all.quantile(0.99)) if len(gan_all)>0 else None,
    'top_outliers_count_p99_plus': int((gan_all > gan_all.quantile(0.99)).sum()) if len(gan_all)>0 else 0
}

# ----- 5. Inspección de outliers y operaciones relevantes -----
if len(df_all) > 0:
    top_gains = df_all.nlargest(20, 'ganancia_num')
    top_losses = df_all.nsmallest(20, 'ganancia_num')
    top_gains.to_csv(ANALYSIS_DIR / "top_20_ganancias.csv", index=False)
    top_losses.to_csv(ANALYSIS_DIR / "top_20_perdidas.csv", index=False)

    p99_val = float(gan_all.quantile(0.99)) if len(gan_all)>0 else None
    p1_val = float(gan_all.quantile(0.01)) if len(gan_all)>0 else None
    outliers = df_all[(df_all['ganancia_num'] >= p99_val) | (df_all['ganancia_num'] <= p1_val)] if p99_val is not None else pd.DataFrame()
    outliers.to_csv(ANALYSIS_DIR / "operaciones_outliers_p1_p99.csv", index=False)

# ----- 6. Guardar resúmenes -----
with open(ANALYSIS_DIR / "analisis_completo_resumen_robusto.json", "w", encoding="utf-8") as f:
    json.dump({'global': global_summary, 'by_group': grouped_results}, f, indent=2, ensure_ascii=False)

pd.DataFrame.from_dict(grouped_results, orient='index').to_csv(ANALYSIS_DIR / "analisis_completo_por_simbolo_periodo_robusto.csv", index=False)

# ----- 7. Graficos: equity por grupo (top N por operaciones) y distribución -----
df_groups = pd.DataFrame.from_dict(grouped_results, orient='index')
if not df_groups.empty:
    top_groups = (df_groups
                  .sort_values('operaciones', ascending=False)
                  .head(6)
                  .apply(lambda r: f"{r['symbol']}_{r['periodo']}", axis=1))

    plt.figure(figsize=(12,8))
    for key in list(top_groups):
        sym, per = key.split("_")
        g = df_all[(df_all['symbol']==sym)&(df_all['periodo']==per)].copy()
        if g.empty:
            continue
        if time_col and time_col in g.columns:
            g = g.sort_values(time_col)
        eq = g['ganancia_num'].cumsum().reset_index(drop=True)
        plt.plot(eq, label=key)
    plt.title("Equity acumulada - top grupos por operaciones")
    plt.legend()
    plt.xlabel("Operación (orden)")
    plt.ylabel("Equity (suma de ganancia)")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "equity_top_groups.png")
    plt.close()

    plt.figure(figsize=(10,5))
    sns.histplot(df_all['ganancia_num'], bins=100, kde=True)
    plt.title("Histograma de ganancia (global)")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "hist_ganancia_global.png")
    plt.close()

    plt.figure(figsize=(8,4))
    sns.boxplot(x=df_all['ganancia_num'])
    plt.title("Boxplot ganancia (global)")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "boxplot_ganancia_global.png")
    plt.close()

# ----- 8. Operaciones robustas revisadas (criterios múltiples) -----
robustos_list = []
for (sym, per), g in df_all.groupby(['symbol','periodo']):
    gan = g['ganancia_num']
    gm = float(gan.mean()) if len(gan)>0 else 0.0
    med = float(gan.median()) if len(gan)>0 else 0.0
    low_iqr = gan.quantile(0.25) if len(gan)>0 else -np.inf
    high_iqr = gan.quantile(0.75) if len(gan)>0 else np.inf
    if 'duracion_num' in g.columns and g['duracion_num'].notna().any():
        dur_ok = g['duracion_num'].between(low_iqr, high_iqr, inclusive="both")
    else:
        dur_ok = True
    cond = (gan > gm) & (gan > med*0.5) & dur_ok
    sel = g[cond]
    robustos_list.append(sel)
robustos_df = pd.concat(robustos_list, ignore_index=True) if robustos_list else pd.DataFrame()
robustos_df.to_csv(ANALYSIS_DIR / "operaciones_robustas_avanzadas.csv", index=False)

# ----- 9. Actualizar reflexiones/logs -----
log_path = LOGS_DIR / "reflexiones_bot.txt"
note = (
    f"\n\n[Auto] Análisis robusto ejecutado para run {RUN_DIR.name}. Archivos guardados en {ANALYSIS_DIR}.\n"
    f"Resumen global: ops={global_summary['operations_total']}, wins={global_summary['wins_total']}, "
    f"winrate={global_summary['winrate']}, pf={global_summary['profit_factor']}\n"
    f"Outliers p99 count: {global_summary.get('top_outliers_count_p99_plus')}\n"
)
existing = log_path.read_text(encoding='utf-8') if log_path.exists() else ""
log_path.write_text(existing + note, encoding='utf-8')

# ----- 10. Salida rápida en notebook -----
print("Análisis robusto completado.")
print("Global summary (breve):", {k: global_summary[k] for k in ('operations_total','wins_total','winrate','profit_factor')})
print(f"Archivos guardados en: {ANALYSIS_DIR}")
print(f"Plots guardados en: {PLOTS_DIR}")
print("Top 5 grupos (por operaciones):")
if not df_groups.empty:
    print(df_groups.sort_values('operaciones', ascending=False).head(5)[['symbol','periodo','operaciones','winrate','profit_factor']])
else:
    print("Sin grupos para mostrar.")

Advertencia: 1680 filas con ganancia no convertible (NaN). Se excluirán de métricas numéricas puntuales.
Fuente primaria cambiada a trades.csv por alto porcentaje de NaN en 'ganancia'.
Análisis robusto completado.
Global summary (breve): {'operations_total': 174, 'wins_total': 103, 'winrate': 0.5919540229885057, 'profit_factor': 1.4322193316101253}
Archivos guardados en: ..\data\backtesting\analisis backtesting 2025-09-06_23-23
Plots guardados en: ..\data\backtesting\analisis backtesting 2025-09-06_23-23\plots
Top 5 grupos (por operaciones):
              symbol periodo  operaciones   winrate  profit_factor
BTCUSDT_30d  BTCUSDT     30d           51  0.588235       1.415318
ETHUSDT_30d  ETHUSDT     30d           48  0.666667       1.732413
BNBUSDT_30d  BNBUSDT     30d           47  0.531915       1.006786
WLDUSDT_30d  WLDUSDT     30d           28  0.571429       1.341317
Análisis robusto completado.
Global summary (breve): {'operations_total': 174, 'wins_total': 103, 'winrate': 0.591954

In [ ]:
# Celda: Auditoría rápida del snapshot consolidado del run (solo este CSV)

import pandas as pd
import numpy as np
from pathlib import Path
import os
from datetime import datetime

# Carpeta de análisis común
def _get_or_create_analysis_dir():
    root = Path("../data/backtesting")
    env_key = "ANALYSIS_BACKTESTING_DIR"
    existing = os.environ.get(env_key)
    if existing:
        d = Path(existing)
        d.mkdir(parents=True, exist_ok=True)
        return d
    stamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
    d = root / f"analisis backtesting {stamp}"
    d.mkdir(parents=True, exist_ok=True)
    os.environ[env_key] = str(d)
    return d
ANALYSIS_DIR = _get_or_create_analysis_dir()
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

SNAP_PATH = Path("../data/backtesting/2025-09-07_00-53/resultados_backtesting_snapshot.csv")
if not SNAP_PATH.exists():
    raise FileNotFoundError(f"No existe el archivo: {SNAP_PATH}")

# Leer
snap = pd.read_csv(SNAP_PATH)
print("Archivo:", SNAP_PATH)
print("Shape:", snap.shape)
print("Columnas:", list(snap.columns))
print("\nTipos de datos (primeras 15):\n", snap.dtypes.head(15))
print("\nPrimeras filas:")
print(snap.head(5))

# Calidad de datos
na_top = snap.isna().sum().sort_values(ascending=False).head(15)
print("\nTop columnas con NaN:\n", na_top)

# Conteos por símbolo y tipo (si existen)
if 'symbol' in snap.columns:
    print("\nOperaciones por símbolo:\n", snap['symbol'].value_counts())
if 'tipo' in snap.columns:
    print("\nDistribución por tipo:\n", snap['tipo'].astype(str).value_counts())

# Intento de conversión de 'ganancia' a numérico y diagnóstico de no convertibles
def _parse_num(s: pd.Series) -> pd.Series:
    s2 = s.astype(str).str.strip()
    s2 = s2.str.replace('%', '', regex=False)
    mask_coma_decimal = s2.str.contains(',', regex=False) & ~s2.str.contains('.', regex=False)
    s2 = s2.where(~mask_coma_decimal, s2.str.replace(',', '.', regex=False))
    s2 = s2.str.replace('\u00A0', '', regex=False)
    return pd.to_numeric(s2, errors='coerce')

if 'ganancia' in snap.columns:
    snap['ganancia_num'] = _parse_num(snap['ganancia'])
    total = len(snap)
    n_ok = int(snap['ganancia_num'].notna().sum())
    n_na = int(total - n_ok)
    print(f"\nConversión 'ganancia' -> numérico: ok={n_ok}, NaN={n_na}, ratio_NaN={n_na/total:.2%}")
    if n_na > 0:
        bad_vals = (snap.loc[snap['ganancia_num'].isna(), 'ganancia']
                        .astype(str)
                        .value_counts()
                        .head(10))
        print("\nTop 10 valores no convertibles en 'ganancia':\n", bad_vals)
else:
    print("\nColumna 'ganancia' no encontrada en snapshot.")

# Métricas simples por símbolo sobre filas con PnL numérico
if ('symbol' in snap.columns) and ('ganancia_num' in snap.columns):
    dfm = snap.dropna(subset=['ganancia_num']).copy()
    if 'tipo' in dfm.columns:
        # opcional: quedarnos con cierres/ventas
        try:
            dfm = dfm[dfm['tipo'].astype(str).str.lower().isin(['venta','sell','close'])]
        except Exception:
            pass
    bysym = []
    for sym, g in dfm.groupby('symbol'):
        gan = g['ganancia_num']
        pos = gan[gan>0].sum()
        neg = gan[gan<=0].abs().sum()
        pf = float(pos/neg) if neg>0 else (np.inf if pos>0 else np.nan)
        winrate = float((gan>0).sum()/len(gan)) if len(gan)>0 else np.nan
        bysym.append({'symbol': sym, 'ops': int(len(gan)), 'winrate': winrate, 'profit_factor': pf})
    if bysym:
        df_bysym = pd.DataFrame(bysym).sort_values('ops', ascending=False)
        print("\nResumen por símbolo (cierres con PnL numérico):")
        print(df_bysym)
        df_bysym.to_csv(ANALYSIS_DIR / "snapshot_resumen_por_simbolo.csv", index=False)

# Guardar una auditoría ligera del snapshot
AUDIT_OUT = ANALYSIS_DIR / "snapshot_audit.csv"
cols_keep = [c for c in ['timestamp','symbol','periodo','tipo','ganancia','ganancia_num','duracion'] if c in snap.columns]
snap[cols_keep].to_csv(AUDIT_OUT, index=False)
print("\nAudit guardado en:", AUDIT_OUT)

Archivo: ..\data\backtesting\2025-09-07_00-53\resultados_backtesting_snapshot.csv
Shape: (2028, 33)
Columnas: ['tipo', 'precio', 'indice', 'timestamp', 'ganancia', 'duracion', 'tipo_mercado', 'indicadores_usados', 'rsi', 'adx', 'votes', 'sl', 'tp', 'signal', 'indicadores_snapshot', 'atr_pct', 'bbw_pct', 'sl_dyn', 'tp_dyn', 'exit_rsi', 'exit_atr', 'exit_adx', 'market_regime', 'equity', 'symbol', 'periodo', 'ganancia_num', 'drawdown_abs', 'drawdown_pct', 'equity_peak', 'drawdown', 'r_multiple', 'outlier_top1']

Tipos de datos (primeras 15):
 tipo                     object
precio                  float64
indice                    int64
timestamp                object
ganancia                float64
duracion                float64
tipo_mercado             object
indicadores_usados       object
rsi                     float64
adx                     float64
votes                   float64
sl                      float64
tp                      float64
signal                   object
indica

In [ ]:
# Celda: Análisis directo de la carpeta ../data/backtesting/resultados (multi-símbolo y periodo)
import pandas as pd
import numpy as np
from pathlib import Path
import os
import re
import json
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Carpeta de análisis común (reusa si ya existe en esta sesión)
def _get_or_create_analysis_dir():
    root = Path("../data/backtesting")
    env_key = "ANALYSIS_BACKTESTING_DIR"
    existing = os.environ.get(env_key)
    if existing:
        d = Path(existing)
        d.mkdir(parents=True, exist_ok=True)
        return d
    stamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
    d = root / f"analisis backtesting {stamp}"
    d.mkdir(parents=True, exist_ok=True)
    os.environ[env_key] = str(d)
    return d

ANALYSIS_DIR = _get_or_create_analysis_dir()
PLOTS_DIR = ANALYSIS_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

RESULT_DIR = Path("../data/backtesting/resultados")
if not RESULT_DIR.exists():
    raise FileNotFoundError(f"No existe la carpeta de resultados: {RESULT_DIR}")

# Utilidades de parsing/limpieza
def parse_numeric_series(s: pd.Series) -> pd.Series:
    s2 = s.astype(str).str.strip()
    s2 = s2.str.replace('%', '', regex=False)
    mask_coma_decimal = s2.str.contains(',', regex=False) & ~s2.str.contains('.', regex=False)
    s2 = s2.where(~mask_coma_decimal, s2.str.replace(',', '.', regex=False))
    s2 = s2.str.replace('\u00A0', '', regex=False)
    return pd.to_numeric(s2, errors='coerce')

def coalesce_profit_cols(df: pd.DataFrame) -> pd.Series:
    candidates = ['ganancia','pnl','PnL','profit','profit_usd','resultado','retorno','retorno_usd','retorno_pct']
    present = [c for c in candidates if c in df.columns]
    if not present:
        return pd.Series([np.nan]*len(df))
    series = None
    for c in present:
        ser = df[c]
        ser_num = parse_numeric_series(ser) if ser.dtype=='O' else pd.to_numeric(ser, errors='coerce')
        series = ser_num if series is None else series.fillna(ser_num)
    return series

def max_drawdown_from_equity(equity: pd.Series) -> float:
    if equity is None or len(equity)==0:
        return 0.0
    eq = pd.Series(equity).astype(float).fillna(method='ffill').fillna(0.0)
    roll_max = eq.cummax()
    dd = (roll_max - eq) / roll_max.replace(0, np.nan)
    return float(dd.fillna(0.0).max())

# Recolectar archivos
files_with_eq = sorted(RESULT_DIR.glob("resultados_*_with_equity.csv"))
files_plain = sorted([p for p in RESULT_DIR.glob("resultados_*.csv") if not p.name.endswith("_with_equity.csv")])
if not files_with_eq and not files_plain:
    raise RuntimeError(f"No se encontraron CSVs en {RESULT_DIR}")

# Regex para extraer symbol y periodo del nombre
pat = re.compile(r"resultados_(?P<symbol>[^_]+)_(?P<periodo>[^_.]+)")

dfs = []
def _extract_meta(p: Path):
    m = pat.search(p.stem)
    sym = m.group('symbol') if m else 'NA'
    per = m.group('periodo') if m else 'NA'
    return sym, per

for p in files_with_eq + files_plain:
    try:
        d = pd.read_csv(p)
        sym, per = _extract_meta(p)
        if 'symbol' not in d.columns:
            d['symbol'] = sym
        if 'periodo' not in d.columns:
            d['periodo'] = per
        # PnL numérico
        d['ganancia_num'] = coalesce_profit_cols(d)
        # Equity: usar si existe, si no construir desde ganancia_num
        eq_col = None
        for cand in ['equity','equity_usd','equity_value']:
            if cand in d.columns:
                eq_col = cand
                break
        if eq_col is not None:
            d['equity_calc'] = pd.to_numeric(d[eq_col], errors='coerce')
        else:
            d['equity_calc'] = d['ganancia_num'].cumsum()
        d['__source__'] = str(p.name)
        dfs.append(d)
    except Exception as e:
        print(f"Error leyendo {p}: {e}")

df_all = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
if df_all.empty:
    raise RuntimeError("No se pudo construir el dataset de análisis desde la carpeta resultados.")

# Filtrar filas con PnL numérico para las métricas
df_all = df_all.dropna(subset=['ganancia_num']).reset_index(drop=True)

# Calcular métricas por símbolo/periodo
def profit_factor_from_series(gan: pd.Series) -> float | None:
    pos = gan[gan>0].sum()
    neg = gan[gan<=0].abs().sum()
    if neg == 0:
        return float('inf') if pos>0 else None
    return float(pos/neg)

group_rows = []
for (sym, per), g in df_all.groupby(['symbol','periodo']):
    gan = g['ganancia_num'].astype(float)
    ops = int(len(gan))
    wins = int((gan>0).sum())
    losses = int((gan<=0).sum())
    denom = ops - int((gan==0).sum())
    winrate = float(wins/denom) if denom>0 else None
    pf = profit_factor_from_series(gan)
    expect = float(gan.mean()) if ops>0 else None
    median = float(gan.median()) if ops>0 else None
    std = float(gan.std()) if ops>0 else None
    mdd = max_drawdown_from_equity(g['equity_calc'])
    group_rows.append({
        'symbol': sym,
        'periodo': per,
        'operaciones': ops,
        'wins': wins,
        'losses': losses,
        'winrate': winrate,
        'profit_factor': pf,
        'expectancy': expect,
        'median': median,
        'std': std,
        'max_drawdown_pct': mdd,
    })

df_groups = pd.DataFrame(group_rows).sort_values(['symbol','periodo']).reset_index(drop=True)
df_groups.to_csv(ANALYSIS_DIR / 'resumen_resultados_por_simbolo_periodo.csv', index=False)

# Resumen global simple
gan_all = df_all['ganancia_num']
global_summary = {
    'operations_total': int(len(gan_all)),
    'wins_total': int((gan_all>0).sum()),
    'losses_total': int((gan_all<=0).sum()),
    'winrate': float((gan_all>0).sum()/len(gan_all)) if len(gan_all)>0 else None,
    'profit_factor': profit_factor_from_series(gan_all),
    'mean': float(gan_all.mean()) if len(gan_all)>0 else None,
    'median': float(gan_all.median()) if len(gan_all)>0 else None,
    'std': float(gan_all.std()) if len(gan_all)>0 else None,
}
with open(ANALYSIS_DIR / 'resumen_global_resultados.json', 'w', encoding='utf-8') as f:
    json.dump(global_summary, f, indent=2, ensure_ascii=False)

# Gráficos rápidos
plt.figure(figsize=(12,7))
top_groups = df_groups.sort_values('operaciones', ascending=False).head(6)
for _, r in top_groups.iterrows():
    sym, per = r['symbol'], r['periodo']
    g = df_all[(df_all['symbol']==sym) & (df_all['periodo']==per)]
    eq = g['equity_calc'].reset_index(drop=True)
    plt.plot(eq, label=f"{sym}_{per}")
plt.title('Equity acumulada - top grupos por operaciones (carpeta resultados)')
plt.xlabel('Operación')
plt.ylabel('Equity (suma o serie)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'equity_top_groups_resultados.png')
plt.close()

plt.figure(figsize=(10,5))
sns.histplot(df_all['ganancia_num'], bins=120, kde=True)
plt.title('Histograma de ganancia - carpeta resultados')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'hist_ganancia_resultados.png')
plt.close()

print('Análisis de carpeta resultados completado.')
print('Resumen global:', global_summary)
print('Archivo resumen por símbolo/periodo:', ANALYSIS_DIR / 'resumen_resultados_por_simbolo_periodo.csv')
print('Plots en:', PLOTS_DIR)

C:\Users\lenovo\AppData\Local\Temp\ipykernel_21048\1485390842.py:59: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  eq = pd.Series(equity).astype(float).fillna(method='ffill').fillna(0.0)
C:\Users\lenovo\AppData\Local\Temp\ipykernel_21048\1485390842.py:59: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  eq = pd.Series(equity).astype(float).fillna(method='ffill').fillna(0.0)
C:\Users\lenovo\AppData\Local\Temp\ipykernel_21048\1485390842.py:59: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  eq = pd.Series(equity).astype(float).fillna(method='ffill').fillna(0.0)
C:\Users\lenovo\AppData\Local\Temp\ipykernel_21048\1485390842.py:59: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ff

Análisis de carpeta resultados completado.
Resumen global: {'operations_total': 648, 'wins_total': 366, 'losses_total': 282, 'winrate': 0.5648148148148148, 'profit_factor': 11.95590026022856, 'mean': 10.941089506172897, 'median': 0.0045000000000000005, 'std': 52.66858448824202}
Archivo resumen por símbolo/periodo: ..\data\backtesting\analisis backtesting 2025-09-08_13-30\resumen_resultados_por_simbolo_periodo.csv
Plots en: ..\data\backtesting\analisis backtesting 2025-09-08_13-30\plots
